# 10 — Error budget, step by step

What the referee asked for: how much uncertainty each pipeline step introduces.
Every section ends with a number that is collected into `products/error_budget.csv`
and rendered into `docs/ERROR_BUDGET.md`.

| step | quantity | how it is estimated here |
|---|---|---|
| CHT lunar limb | radius and centre scatter | distribution over all frames; within‑group scatter after removing the tracking drift; current vs 2024 scikit‑image |
| celestial North | angle error | star centroid precision, PSF width, star‑pair baseline (notebook 03) |
| timestamps / ephemeris | Sun–Moon vector error | 1 s clock quantisation × relative motion |
| Sun centre | relative and absolute error | propagation of the above; inter‑position registration measured on the stacks; star‑based absolute check |
| alignment + stacking | residual misregistration, interpolation | frame‑to‑stack residuals, sub‑pixel shift smoothing |
| HDR | linearity / saturation | exposure ratios, LDIC gains |
| polarisation angle | angle error | = North‑angle error ⊕ P‑angle precision |

In [ ]:
import sys, time
sys.path.insert(0, "..")          # config.py / utils.py live one level up
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

import config, utils
%matplotlib inline

In [ ]:
budget = []
def add(step, quantity, value, unit, how):
    budget.append(dict(step=step, quantity=quantity, value=float(value), unit=unit, how=how))
    print(f"[{step}] {quantity}: {float(value):.3g} {unit}")

table = pd.read_csv(config.FRAME_TABLE_CSV, dtype={"position": str})
centers = pd.read_csv(config.SUN_MOON_CENTERS_CSV)
pol = table[table.position.isin(config.POLARIZER_POSITIONS)].copy()
inv = pd.read_csv(config.PRODUCTS_DIR / "frame_inventory.csv", dtype={"position": str}).set_index("filename")
pol["t"] = pd.to_datetime(inv.loc[pol.filename, "date_obs_utc"].values)
cs = pd.read_csv(config.PRODUCTS_DIR / "celestial_north_summary.csv").iloc[0]
scale = float(cs.scale_pair_mean)          # arcsec/px from the star separation (translation-free)
rsun_px = 959.0 / scale
print(f"{len(pol)} polarimetric frames; plate scale {scale:.4f} arcsec/px")

## 1. Lunar limb (CHT)

In [ ]:
mean_r, std_r = pol.moon_radius.mean(), pol.moon_radius.std(ddof=0)
fig, ax = plt.subplots(figsize=(8, 4.5))
bins = np.arange(np.floor(pol.moon_radius.min()) - 0.5, np.ceil(pol.moon_radius.max()) + 1.5, 1)
for pos, col in zip(config.POLARIZER_POSITIONS, ("red", "green", "blue")):
    ax.hist(pol[pol.position == pos].moon_radius, bins=bins, histtype="stepfilled", alpha=0.35, color=col,
            label=f"pol{pos} ({config.POLARIZER_ANGLE_DEG[pos]:+.0f}°)".replace("+0°", "0°"))
ax.axvline(mean_r, color="k", lw=1.5, label=f"mean = {mean_r:.1f} px")
ax.axvspan(mean_r - std_r, mean_r + std_r, color="grey", alpha=0.15, label=f"±1σ = {std_r:.1f} px")
for r in pol[~pol.radius_ok].moon_radius:
    ax.plot(r, 0.3, "kx", ms=8)
ax.plot([], [], "kx", label="rejected frames")
ax.set_xlabel("CHT lunar radius [px]"); ax.set_ylabel("frames"); ax.legend(fontsize=9); ax.grid(alpha=0.3)
ax.set_title("Circular-Hough lunar radius over the 136 polarimetric frames")
fig.savefig(config.FIGURES_DIR / "error_cht_moon_radius_hist.png", dpi=200, bbox_inches="tight")
add("CHT", "lunar radius scatter, all frames (1 sigma)", std_r, "px", "std of the CHT radius over 136 frames (the paper's 1-sigma rejection filter)")
add("CHT", "lunar radius scatter, all frames, relative", std_r / mean_r * 100, "%", "propagates directly into the per-frame plate scale")

In [ ]:
# Within-group scatter: radius, and centre after removing the linear tracking drift within each (pol, exposure) group
good = pol[pol.use].copy()
res_r, res_x, res_y = [], [], []
for (p, e), g in good.groupby(["position", "inverse_exposure_time"]):
    if len(g) < 3: continue
    t = (g.t - g.t.min()).dt.total_seconds().values
    res_r += list(g.moon_radius - g.moon_radius.mean())
    for col, store in (("moon_xc", res_x), ("moon_yc", res_y)):
        fit = np.polyfit(t, g[col].values, 1)
        store += list(g[col].values - np.polyval(fit, t))
add("CHT", "lunar radius scatter within a stack group (1 sigma)", np.std(res_r), "px", "frames of the same polariser+exposure, 10-15 s apart")
add("CHT", "Moon-centre scatter within a group, x (1 sigma)", np.std(res_x), "px", "after removing the linear tracking drift")
add("CHT", "Moon-centre scatter within a group, y (1 sigma)", np.std(res_y), "px", "after removing the linear tracking drift")
print(f"tracking drift during a group: up to {good.groupby(['position', 'inverse_exposure_time']).moon_yc.agg(np.ptp).max():.0f} px")

In [ ]:
# Reproducibility of the CHT itself: current scikit-image run (notebook 02) vs the 2024 run used for the paper
if config.MOON_CENTERS_CSV.exists():
    new = pd.read_csv(config.MOON_CENTERS_CSV).set_index("Filename")
    leg = pd.read_csv(config.LEGACY_MOON_CENTERS_CSV); leg["Filename"] = leg.Filename.str.split("/").str[-1]
    leg = leg.drop_duplicates("Filename").set_index("Filename")
    j = new.join(leg, rsuffix="_legacy", how="inner")
    dxy = np.hypot(j.Moon_XC - j.Moon_XC_legacy, j.Moon_YC - j.Moon_YC_legacy)
    gross = dxy > 10
    for c in ("Moon_XC", "Moon_YC", "Moon_Radius"):
        d = (j[c] - j[c + "_legacy"])
        print(f"{c:12s}: differs on {(d != 0).sum()}/{len(d)} frames, max |diff| {d.abs().max():.0f} px")
    print(f"gross failures (> 10 px) on {gross.sum()} frames: {list(j.index[gross])}")
    add("CHT", "CHT reproducibility across scikit-image versions, typical (rms, excluding gross failures)", np.sqrt(np.mean(dxy[~gross] ** 2)), "px", "notebook 02 (2026) vs the 2024 table, centre distance")
    add("CHT", "CHT gross failures (> 10 px) between library versions", int(gross.sum()), "frames", "of 145; these are the frames with an inconsistent lunar radius, which the 1-sigma filter removes")
else:
    print("products/moon_centers.csv not found — run notebook 02")

## 2. Celestial North from the stars (notebook 03)

In [ ]:
cn = pd.read_csv(config.CELESTIAL_NORTH_FIT_CSV)
fwhm = float(cs.fwhm_px)
baseline = 1500.0    # px between zeta Psc and 88 Psc
snr = float(cn.min_snr.mean())
sigma_c = fwhm / (2.3548 * snr)                                  # centroid precision of one star
add("North", "PSF FWHM of the stars", fwhm, "px", "2-D Gaussian fits on the 1/3 s frames")
add("North", "PSF FWHM of the stars", fwhm * scale, "arcsec", "")
add("North", "statistical angle error from star centroids", np.degrees(np.sqrt(2) * sigma_c / baseline), "deg", "sqrt(2) x sigma_centroid / baseline (1500 px)")
add("North", "angle error of the manual Stellarium overlay", np.degrees(fwhm / baseline), "deg", "one PSF width over the star baseline (protractor on a screenshot)")
add("North", "frame-to-frame scatter of the fitted angle (1 sigma)", cn.alpha_pair.std(ddof=1), "deg", "translation-free star-pair fits on 6 frames")
add("North", "fitted angle minus paper value (168 deg)", cs.alpha_pair_mean - config.CELESTIAL_NORTH_DEG, "deg", "star-pair mean")
add("North", "plate scale from the stars", cs.scale_pair_mean, "arcsec/px", f"paper quotes the optical value {config.PLATE_SCALE_ARCSEC_PER_PIX}")
cn.round(3)

## 3. Timestamps and ephemeris

In [ ]:
from astropy.time import Time
import astropy.units as u
eph = utils.Ephemeris()
utc0 = config.TOTALITY_MID_UTC
g0 = eph.sun_moon_geometry(utc0); g1 = eph.sun_moon_geometry((Time(utc0) + 10 * u.s).isot)
v0 = np.array(utils.sky_offset_to_pixels(g0[2], g0[3], 1 / scale)); v1 = np.array(utils.sky_offset_to_pixels(g1[2], g1[3], 1 / scale))
rate = np.hypot(*(v1 - v0)) / 10
add("time", "Sun-Moon relative motion", rate * scale, "arcsec/s", "Skyfield, mid-totality")
add("time", "Sun-centre error from the 1 s camera-clock quantisation", rate * 0.5, "px", "half a second x relative rate")
print(f"(a camera-clock offset would move the Sun-Moon vector by {rate:.2f} px per second of offset)")

## 4. Sun centre: propagation, inter-position registration, absolute check

In [ ]:
sep_px = float(centers.merge(pol[["filename"]], on="filename").sep_arcsec.mean() / scale)
d_scale = std_r / mean_r
d_alpha = fwhm / baseline
add("Sun", "Sun-Moon separation during totality", sep_px, "px", "mean over frames (27-40 arcsec)")
add("Sun", "Sun-centre error from the plate-scale scatter", sep_px * d_scale, "px", "sep x (delta scale / scale)")
add("Sun", "Sun-centre error from the North-angle error", sep_px * d_alpha, "px", "sep x delta alpha")
add("Sun", "Sun-centre error from the Moon-centre scatter", np.hypot(np.std(res_x), np.std(res_y)), "px", "dominant random term")

# convention offset (legacy vs astrometric) per polariser position
rows = []
for _, r in centers[[utils.position_from_filename(f) in config.POLARIZER_POSITIONS for f in centers.filename]].iterrows():
    sx_l, sy_l, _ = utils.sun_center_from_moon(g0[0], r.sep_arcsec, r.pa_sun_from_moon_deg, 0, 0, r.moon_radius, convention="legacy")
    sx_a, sy_a, _ = utils.sun_center_from_moon(g0[0], r.sep_arcsec, r.pa_sun_from_moon_deg, 0, 0, r.moon_radius, convention="astrometric")
    rows.append(dict(position=utils.position_from_filename(r.filename), dx=sx_l - sx_a, dy=sy_l - sy_a))
conv = pd.DataFrame(rows).groupby("position").mean()
conv["rel_to_pol2_dx"] = conv.dx - conv.loc["2", "dx"]; conv["rel_to_pol2_dy"] = conv.dy - conv.loc["2", "dy"]
conv["legacy_by_eye_shift (dy,dx)"] = [str(config.LEGACY_CHANNEL_SHIFTS[p]) for p in conv.index]
print("legacy - astrometric Sun centre (px), and the by-eye shifts that were used downstream:")
display(conv.round(1))
add("Sun", "inter-position offset induced by the legacy convention (max)", np.hypot(conv.rel_to_pol2_dx, conv.rel_to_pol2_dy).max(), "px", "pol1/pol3 relative to pol2; compensated by the by-eye channel shifts in the submitted paper")

In [ ]:
# Inter-polariser registration measured directly on the stacks: cross-correlation of DoG-filtered
# patches around the Sun at 1.4 Rsun (median of 12 patches; the 1/13 s stacks have the best SNR)
reg = []
for e in (13, 25):
    ours = fits.getdata(utils.stacked_filename(e), memmap=True)
    for pos, i in (("1", 0), ("3", 2)):
        dy, dx, spread = utils.measure_interchannel_shift(ours[1], ours[i], rsun_px=rsun_px)
        reg.append(dict(source=f"this pipeline ({config.SUN_CENTER_ROTATION})", inv_exp=e, pol=pos, dy=dy, dx=dx, patch_spread=spread))
    if config.LEGACY_STACKED_DIR.exists():
        leg = fits.getdata(config.LEGACY_STACKED_DIR / f"Linear_Composite_{e}.fits", memmap=True)
        for pos, i in (("1", 0), ("3", 2)):
            dy, dx, spread = utils.measure_interchannel_shift(leg[1], leg[i], rsun_px=rsun_px)
            reg.append(dict(source="legacy stacks (paper), before the by-eye shifts", inv_exp=e, pol=pos, dy=dy, dx=dx, patch_spread=spread))
reg = pd.DataFrame(reg); reg["abs"] = np.hypot(reg.dx, reg.dy)
display(reg.round(2))
for src, v in reg.groupby("source")["abs"].mean().items():
    add("Sun", f"measured pol1/pol3 vs pol2 misregistration: {src}", v, "px", "cross-correlation of DoG-filtered patches at 1.4 Rsun, 1/13 + 1/25 s stacks")

In [ ]:
# Absolute check from the star field (notebook 03): with the North angle and plate scale fitted, the stars
# sit ~2 px rms from their predicted positions relative to the CHT Moon centre -> the Moon centre (and hence
# the Sun centre, 27-40" away) is right at that level.  The CHT lunar radius is ~0.5 % too small (a scale
# bias, harmless for the Sun centre but relevant for arcsec / Rsun scale bars).
add("Sun", "absolute Moon/Sun-centre accuracy (star-field check, rms)", float(cs.fit_rms_px), "px", "residual of 2 stars after fitting alpha and plate scale, Moon fixed at the CHT centre")
add("Sun", "absolute Moon/Sun-centre accuracy (star-field check, rms)", float(cs.fit_rms_px) * scale, "arcsec", "")
add("Sun", "CHT lunar-radius bias (plate scale from lunar radius vs from stars)", 100 * (cs.scale_moon_mean / cs.scale_pair_mean - 1), "%", "the CHT circle is ~3-4 px too small; use 1.50 arcsec/px")

## 5. Alignment and stacking

In [ ]:
# Frame-to-stack residual in the 1.2-2 Rsun annulus (4x4 binned)
yy, xx = np.mgrid[:config.IMAGE_SHAPE[0]:4, :config.IMAGE_SHAPE[1]:4]
rr = np.hypot(xx - config.SUN_CENTER_XY[0], yy - config.SUN_CENTER_XY[1]) / rsun_px
ann = (rr >= config.ERROR_ANNULUS_RSUN[0]) & (rr <= config.ERROR_ANNULUS_RSUN[1])
rows = []
cen = good.set_index("filename")
for e in (13, 50, 200):
    stack = fits.getdata(utils.stacked_filename(e), memmap=True)
    for i, pos in enumerate(config.POLARIZER_POSITIONS):
        files = list(good[(good.position == pos) & (good.inverse_exposure_time == e)].filename)
        S = np.asarray(stack[i][::4, ::4], np.float64)
        for f in files:
            L = utils.luminance(fits.getdata(config.CALIBRATED_LIGHTS_DIR / f).astype(np.float32))
            F = utils.shift_to_center(L, cen.loc[f, "sun_xc"], cen.loc[f, "sun_yc"])[::4, ::4].astype(np.float64)
            rel = (F - S)[ann] / (S[ann] + 1e-9)
            rows.append(dict(inv_exp=e, pol=pos, frame=f, rel_rms=float(np.sqrt(np.mean(rel ** 2))), n=len(files)))
fr = pd.DataFrame(rows)
display(fr.groupby(["inv_exp", "pol"]).rel_rms.mean().unstack().round(4))
add("stack", "frame-to-stack relative rms in 1.2-2 Rsun (1/50 s)", fr[fr.inv_exp == 50].rel_rms.mean(), "", "photon noise + any misregistration; a pure-noise stack of n frames gives sqrt(1-1/n) x single-frame noise")
add("stack", "bilinear sub-pixel shift: worst-case smoothing", 0.5, "px", "interpolation kernel half-width")

## 6. HDR

In [ ]:
rt = pd.read_csv(config.PRODUCTS_DIR / "exposure_ratio_table.csv", index_col=0)
display(rt.round(3))
lin = rt.iloc[:4][list(config.POLARIZER_POSITIONS)].values                     # the four shortest steps, unsaturated
add("HDR", "exposure-ratio deviation from nominal (unsaturated steps, rms)", np.sqrt(np.mean((lin / rt.expected.values[:4, None] - 1) ** 2)) * 100, "%", "max(short)/max(long) vs t_short/t_long, 1/800..1/50 s")
sat = int((rt[list(config.POLARIZER_POSITIONS)] > 0.85).sum().sum())
add("HDR", "bracket steps flagged as saturated", sat, "steps", "ratio > 0.85, of 8 steps x 3 polarisers")
ld = np.load(config.PRODUCTS_DIR / "ldic_diagnostics.npz")
k = ld["k_at_pol1"]
gain_dev = [np.std(k[j]) / np.mean(k[j]) for j in range(1, len(k)) if np.mean(k[j]) > 0]
add("HDR", "LDIC gain variation with position angle (rms/mean, pol1)", np.mean(gain_dev) * 100, "%", "spread of k_i(phi) across the 60 sectors")

## 7. Polarisation angle

In [ ]:
from sunpy.coordinates import sun
P = float(sun.P(Time(config.TOTALITY_MID_UTC)).to("deg").value)
add("pol", "P-angle recomputed minus paper value", P - config.SOLAR_P_ANGLE_DEG, "deg", "SunPy")
add("pol", "polarisation-angle zero-point error", np.degrees(fwhm / baseline), "deg", "= North-angle error of the manual overlay; the polariser mounting itself is not calibrated here")

## Summary

In [ ]:
eb = pd.DataFrame(budget)
eb.to_csv(config.ERROR_BUDGET_CSV, index=False)
md = ["# Error budget", "", "_Generated by `notebooks/10_error_budget.ipynb` — do not edit by hand._", "",
      f"Run: {time.strftime('%Y-%m-%d')} · Sun-centre convention `{config.SUN_CENTER_ROTATION}` · plate scale {scale:.4f} arcsec/px", "",
      "| step | quantity | value | unit | how |", "|---|---|---|---|---|"]
for _, r in eb.iterrows():
    md.append(f"| {r.step} | {r.quantity} | {r.value:.3g} | {r.unit} | {r.how} |")
md += ["", "Figures: `figures/error_cht_moon_radius_hist.png`, `figures/celestial_north_star_cutouts.png`, "
       "`figures/celestial_north_overlay.png`, `figures/ldic_weights_and_gains.png`."]
(config.DOCS_DIR / "ERROR_BUDGET.md").write_text("\n".join(md))
eb